In [11]:
import pandas as pd
import json
import random
import re

def load_log(csv_path): #파일 읽기
    df = pd.read_csv(csv_path)

    def extract_msg(logstr):
        try: 
            outer = json.loads(logstr) #원본 로그가 이중구조로 되어있기 때문에, logstr에 저장되어있던 "Log"열dmf outer에 저장.
            #원본 로그 중 한줄(nezha) / "Log"열.
            # "{""log"":""
            # {\""message\"":\""TraceID: 0163a3c609fab47d57e071bc12344f25 SpanID: 0f6a63f0464e170f 
            # Query product with name and description successfully\"",
            # \""severity\"":\""info\"",\""timestamp\"":\""2022-08-22T03:51:10.529109633Z\""}\n"",
            # ""stream"":""stdout"",""time"":""2022-08-22T03:51:10.529207382Z""}"
            inner = json.loads(outer["log"]) # outer에 저장되어있던 "Log"열중 log이름이 붙은 데이터를 꺼냄.
            return inner.get("message", "") # inner에 메시지 부분 리턴.
        except Exception: 
            return "" #실패시 빈 문자열 리턴. 종료.

    df["msg"] = df["Log"].apply(extract_msg) 
    #"Log"열만 선택. apply는 Log열 데이터를 extract_msg에 전달
    # msg열에 저장. (상자 해체만 한 상태.)
    
    df["service"] = df["PodName"].str.split("-").str[0] 
    # "PodName"열 예시 --> productcatalogservice-668d5f85fb-wckp8
    # "PodName"열의 productcatalogservice만 가져와 service라는 새로운 열에 저장.

    df["event"] = df["msg"].str.replace(r"TraceID:\s*\S+\s+SpanID:\s*\S+\s*","", regex=True).str.strip()
    # TraceID: 0163a3c609fab47d57e071bc12344f25 / SpanID: 0f6a63f0464e170f
    # Query product with name and description successfully(뒤에 바로 있어서 앞의 문자들이 지워지면 자연스럽게 공백을 지워서 나타남.)
    # replace하므로, traceid/spanid는 사라짐. 결국 메시지 원문을 남기기 위한 작업. 메시지는 "event"열에 저장.
    
    
    return df    # 데이터 리턴.

In [12]:
def find_cause_lines(df, cause_service, cause_pattern): #정의된 원인 서비스와 이름 넘겨받음
    hit = df[(df["service"] == cause_service) & (df["msg"].str.contains(re.escape(cause_pattern), na=False))]
    # na(비어있는 행은 false처리) re.escape(특수기호 쳐내기) --> 조건에 맞는 것들만 필터링
    return hit.index.tolist() #해당 로그들이 원본파일에 몇번째 줄에 있는지 정리.(리스트로)

def make_distractors(df, correct_service, k = 4, seed = 42): #오답지선다 만들기.
    rng = random.Random(seed)
    pool = []
    for svc, g in df.groupby("service"):
        if svc == correct_service: #정답 걸러내기. 
            continue 
        
        vc = g["event"].value_counts() # 이벤트가 얼마나 많은지 세기 --> 제일 많이 나온 오류를 찾는것.(그럴싸한 오답 만들어내기.)
        if len(vc) > 0: # 0보다 크면, 제일 많이 나왔던것 저장.
            top = vc.index[0]
        else:
            top = None # none
            
        if top: #오답 담아내기. pool에 저장.
            pool.append((svc, top)) 
    rng.shuffle(pool) # 각 서비스마다 빈도가 많은것들 섞기.(groupby 한것들 마다 빈도수 측정.)
    return pool[: k-1] # 섞은후 리스트로 필요한 개수만큼 반환. 

In [13]:
def build_question(df, cause_service, cause_pattern, k =4, seed = 42):
    correct_event = cause_pattern
    distractors = make_distractors(df, cause_service, k = k, seed = seed)
    
    options = [(cause_service, correct_event)] + distractors #리스트 형태로 4지선다.(정답 서비스, 정답 이벤트) + 3개의 오답(오답 서비스, 오답 이벤트)
    rng = random.Random(seed)
    rng.shuffle(options) # 4지선다 섞기
    
    correct_idx = None
    for i, (svc,ec) in enumerate(options):
        if svc == cause_service:
            correct_idx = i
            break
    #보기에 번호 붙이고, 진짜 정답 찾아두기(correct_idx에 저장)
    
    return {
        "options" : [f"{svc} : {ev}" for svc, ev in options],
        "correct_idx" : correct_idx,
        "correct_label" : f"{cause_service}: {correct_event}"
    }
    
    
def build_prompt(log_text, question):
    opts = "\n".join( #줄바꾸면서 엮기
        f"{i+1}. {opt}" for i,opt in enumerate(question["options"])
    )
    return (
        "다음은 마이크로서비스 시스템의 로그입니다. "
        "이 로그에 나타난 장애의 근본 원인(어느 서비스의 어떤 이벤트)을 고르세요.\n"
        "반드시 보기 번호 하나만 숫자로 답하세요.\n\n"
        f"[로그(압축기 결과 자리)]\n{log_text}\n\n"
        f"[보기]\n{opts}\n\n"
        "정답 번호:"
    )
    
def grade(llm_answer_text, question):
    m = re.search(r"\d+", str(llm_answer_text)) #llm이 내뱉은 말에서 /d+를 이용해 숫자만 골라냄.
    if not m: #답변을 못고른경우
        return 0
    picked = int(m.group()) - 1 #정수로 바꾼후, 리스트 순서에 맞게 -1
    return int(picked == question["correct_idx"]) #아까 고른 정답과 같으면 T, 아니면 F

In [14]:
def compress(df, method="none", keep_lines=None):
    if method == "none" or keep_lines is None: #압축하지 않거나, 보존 라인이 없는경우
        return df #그대로 원본 데이터 반환.
    return df.loc[sorted(keep_lines)] # 줄 번호들을 오름차순으로 만들고, 번호에 해당하는 행들만 골라 새로운 표

def df_to_logtext(df, max_lines = None):
    lines = df["event"].tolist()
    if max_lines: #최대 라인 설정.
        lines = lines[:max_lines]
    return "\n".join(lines)

In [ ]:
if __name__ == "__main__":
    CSV =  "03_51_log.csv"
    
    CAUSE_SVC = "productcatalogservice" # 범인 서비스 이름 정의
    CAUSE_PAT = "Query product with name and description successfully" # 에러 문구 정의
    
    df = load_log(CSV)
    print(f"로그 {len(df)}줄 로드, 서비스 {df['service'].nunique()}개")
    
    cause_idx = find_cause_lines(df, CAUSE_SVC, CAUSE_PAT) #cause_servie, cause_pattern
    print(f"정답(원인) 줄 {len(cause_idx)}개 발견")

    q = build_question(df, CAUSE_SVC, CAUSE_PAT, k=4, seed=42)
    print("\n== 생성된 k지선다 문제 ===")3
    
    for i,opt in enumerate(q["options"]):
        mark = " (정답)" if i == q["correct_idx"] else ""
        print(f" {i+1}. {opt}{mark}")
        
    kept = compress(df, method="none")
    log_text = df_to_logtext(kept, max_lines=40)
    prompt = build_prompt(log_text, q)
    print("\n== LLM 프롬프트 (앞부분) ==")
    print(prompt[:600], "...")
    
    fake_answer = str(q["correct_idx"] + 1)
    print(f"\n채점 데모: LLM이 '{fake_answer}' 답 -> 점수 {grade(fake_answer, q)}")

로그 25574줄 로드, 서비스 10개
정답(원인) 줄 2418개 발견

== 생성된 k지선다 문제 ===
 1. frontend : Request complete
 2. currencyservice : Handles decimal or fractional carrying
 3. paymentservice : Transaction processed: visa ending 0454     Amount: CAD101.31755855
 4. productcatalogservice : Query product with name and description successfully (정답)

== LLM 프롬프트 (앞부분) ==
다음은 마이크로서비스 시스템의 로그입니다. 이 로그에 나타난 장애의 근본 원인(어느 서비스의 어떤 이벤트)을 고르세요.
반드시 보기 번호 하나만 숫자로 답하세요.

[로그(압축기 결과 자리)]
Query product with name and description successfully
Query product with name and description OLJCESPC7Z
Request complete
Adding to cart complete
Adding to cart started
Request started

[GetQuote] completed request
[CreateQuoteFromCount] completed request
[CreateQuoteFromFloat] completed request
[CreateQuoteFromFloat] received request
[QuoteByCountFloat] completed request
[QuoteByCountFloat] received request
[CreateQuoteFromCount] received request
[GetQuote] received request
List Recommen ...

채점 데모: LLM이 '4' 답 -> 점수 1
